Wir müssen eine Presentation fertig haben. 

In drei Wochen ist ein Feiertag. 
Wir machen ein Treffen um das Projekt weiter zu machen.


REQ-FUNC-001:

INMP441    →  Raspberry Pi Zero GPIO
VDD        →  Pin 1  (3.3V)
GND        →  Pin 6  (GND)
SCK        →  Pin 12 (GPIO18 - Bit Clock)
WS         →  Pin 35 (GPIO19 - Word Select / LRCLK)
SD         →  Pin 38 (GPIO20 - Serial Data)
L/R        →  GND    (Linker Kanal)

Rapsberry /boot/config.txt:
dtparam=i2s=on
dtoverlay=i2s-mmap

/ect/asound.conf:
pcm.i2smic {
    type hw
    card 0
}

pcm.!default {
    type plug
    slave.pcm "i2smic"
}

In [ ]:
import sounddevice as sd # wrapper basiert auf libportaudio2 (C-Bib)
# Vielleicht ist es hier besser was eigenes zu machen??
with sd.InputStream(
    samplerate=SAMPLERATE,
    blocksize=BLOCKSIZE,
    dtype=DTYPE,
    channels=CHANNELS,
    callback=audio_callback,
    device='hw:0,0'   # I2S Device
):
    print("Aufnahme läuft...")
    sd.sleep(int(DURATION_MIN * 60 * 1000))

ModuleNotFoundError: No module named 'sounddevice'

REQ-FUNC-002:

Das hier ist eine Standard Programmieraufgabe. Vielleicht ein Queue nutzen um die Last der GUI aus dem Thread der Verarbeitung des Audiosignales rauszuhalten?
  

REQ-FUNC-003:

Hier sollten wir nicht unbedingt einen Pythonloop verwenden, sondern einen C-Loop in einem Python-Modul. Z.b. mit Vektorisierung über Numpy. 

FÜr diese Anforderung kann auch der Zero 2 Sinvoll sein. 

REQ-FUNC-004

Klassische Coding Aufgaben. Sollte kein Problem sein. 

REQ-FUNC-005

Klassische Coding Aufgaben. Sollte kein Problem sein. 

REQ-FUNC-006

Webbasierte GUI
Vielleicht Fast API
oder Streamlit

REQ-FUNC-007

Hier lässt sich schonmal ein kleiner Testcode schreiben. Vielleicht Dataclass.

REQ-FUNC-008

Hier könnte ein Button und eine Status LED am Gerät sehr praktisch sein. 
Auch eine Commandline ist hier von Vorteil.



REQ-FUNC-009


Architektur: Drei Ebenen
Ebene 1: Der Hauptprozess (Hartzeitrechner) Dieser Prozess läuft auf dem Raspberry Pi und hat zwei Aufgaben: Er verarbeitet das Audiosignal in Echtzeit (I2S → SPL-Berechnung) und stellt gleichzeitig eine programmatische Schnittstelle bereit. Diese Schnittstelle muss extrem stabil sein – wenn die Audio-Verarbeitung blockiert, entstehen Messfehler. Daher läuft die API in einem separaten Thread oder Prozess, der nur lesend auf die berechneten Werte zugreift, niemals schreibend in den Audiodatenstrom eingreift.

Ebene 2: Die API (Kommunikationsbrücke) Die API ist die einzige Stelle, an der externe Steuerbefehle empfangen werden. Sie definiert eine minimale Sprache: Zustandsübergänge (Start/Stop), Abfragen (aktueller Pegel, Leq-Wert), und Konfiguration (Kalibrierfaktor setzen, A- vs. Z-Bewertung wählen). Technisch ist dies ein HTTP-Server, aber es könnte auch eine Message Queue oder ein Unix-Socket sein. Die Wahl fiel auf HTTP, weil die Web-GUI (REQ-FUNC-006) diese Schnittstelle ohnehin benötigt – man implementiert sie also nur einmal und nutzt sie zweimal.

Ebene 3: Der CLI-Client (dünne Schicht) Dies ist ein separates Programm, das nicht auf dem Pi laufen muss, aber kann. Es versteht menschliche Befehle ("starte Messung", "wie ist der Status?") und übersetzt diese in die technischen API-Aufrufe. Der Client hat keinen eigenen Zustand – er fragt nur an und zeigt Ergebnisse an. Das macht ihn wartungsarm: Änderungen an der Messlogik erfordern keine Änderung am Client, solange die API stabil bleibt.

Konnektivitätsoptionen
Fernzugriff via SSH Der Raspberry Pi Zero W bringt WiFi mit. Standardmäßig aktiviert Raspberry Pi OS einen SSH-Server. Ein Entwickler oder Techniker verbindet sich mit ssh pi@spl-meter.local, landet in einer Shell und führt dort den CLI-Client aus. Dies erfordert keinerlei zusätzliche Software auf dem Pi, nutzt bestehende Linux-Infrastruktur.

Lokaler Zugriff via USB/seriell Für Umgebungen ohne WLAN oder wenn der Pi direkt an einen Laptop angeschlossen wird, nutzt man den "USB Gadget Mode". Der Pi Zero meldet sich am USB-Port des Host-Computers nicht als Massenspeicher, sondern als serielles Gerät (virtueller COM-Port) an. Über diese serielle Verbindung erhält man direkten Zugriff auf die Linux-Konsole des Pi. Technisch gesehen ist dies identisch zu SSH, nur dass das physische Kabel statt WiFi als Übertragungsmedium dient. Das CLI-Tool funktioniert identisch, egal ob man über SSH oder USB/seriell verbunden ist.

Netzwerkzugriff via HTTP direkt Fortgeschrittene Nutzer oder externe Systeme können die API auch direkt ansprechen, ohne das CLI-Tool zu verwenden. Dies ermöglicht Integration in andere Software, automatisierte Tests, oder Fernsteuerung aus der Cloud (wenn entsprechende Sicherheitsmaßnahmen implementiert sind).

Vorteile dieser Herangehensweise
Wiederverwendung und Konsistenz Weil Web-GUI und CLI dieselbe API nutzen, ist garantiert, dass beide Steuerungswege identische Ergebnisse liefern. Ein Fehler in der API wird sofort in beiden Oberflächen sichtbar, was die Fehlersuche vereinfacht. Neue Funktionen müssen nur an einer Stelle implementiert werden.

Trennung der Belange Die Audio-Echtzeitverarbeitung (hartes Echtzeitsystem) und die Benutzersteuerung (weiches System mit Netzwerkverzögerungen) sind strikt getrennt. Das CLI kann blockieren, langsam reagieren oder abstürzen – die Messung läuft unbeeinflusst weiter. Dies ist kritisch für Messgenauigkeit.

Testbarkeit Die API kann unabhängig vom gesamten System getestet werden. Man kann mit Tools wie curl oder Postman Befehle senden und prüfen, ob die Zustandsmaschine korrekt reagiert, ohne ein Mikrofon anschließen zu müssen.

Alternative Ansätze (und warum sie hier nicht gewählt wurden)
Man könnte die Steuerung auch über eine direkte serielle Kommunikation zwischen PC und Pi implementieren (z.B. ein eigenes Binärprotokoll). Dies wäre ressourcenschonender, aber erfordert einen eigenen Treiber auf dem PC-Seite und funktioniert nicht nativ über SSH.

Eine Shared-Memory-Lösung auf dem Pi allein (ohne Netzwerk) wäre noch schneller, aber schließt Fernzugriff aus und macht das System schwerer testbar.

Die gewählte HTTP-basierte Lösung ist der sweet spot zwischen Einfachheit (keine Treiber, kein Binärprotokoll), Flexibilität (SSH, USB, WiFi funktionieren alle) und Wartbarkeit (Standardwerkzeuge, dokumentierte Schnittstelle).


REQ-FUNC-010
Das Problem: Zwei unterschiedliche Zeitskalen
Die Audio-Verarbeitung ist ein hartes Echtzeitsystem. Der Mikrofon-Chip liefert alle 20 Mikrosekunden ein Sample. Wenn der Prozess zu langsam antwortet, entsteht ein Dropout – die Messung ist unbrauchbar. Schreiboperationen auf eine SD-Karte dagegen sind langsam und unvorhersehbar: Manchmal dauert es Millisekunden, manchmal hunderte Millisekunden, wenn die Karte intern reorganisiert. Man kann nicht beides in derselben Zeitachse erledigen.

Die Lösung: Entkopplung durch einen Puffer
Das System nutzt einen Producer-Consumer-Ansatz mit einem Ringpuffer dazwischen:

Der Producer (Audio-Callback): Läuft im hochpriorisierten Echtzeit-Thread. Er empfängt den Audiodatenblock vom I2S-Interface und legt ihn non-blocking in einen speicherbasierten Ringpuffer ab. Das dauert Mikrosekunden. Wenn der Puffer voll ist (weil die SD-Karte gerade langsam ist), verwirft er die ältesten Daten oder den neuen Block – je nach Strategie – aber er blockiert niemals.

Der Consumer (Schreib-Thread): Läuft in einem separaten, niedrigpriorisierten Thread. Er liest aus dem Ringpuffer, blockiert dabei gerne (er hat keine Echtzeit-Anforderung), und schreibt die Daten in Ruhe auf die SD-Karte. Wenn die Karte mal hängt, läuft der Puffer voll, aber die Audio-Erfassung läuft unbeeinträchtigt weiter.

Der Dateihandling: Streaming statt Sammeln
Die Audiodatei wird im WAV-Format geschrieben, das für Streaming geeignet ist. Der Header mit den Metadaten (Abtastrate, Bittiefe, Kanäle) wird am Anfang platziert, dann werden die Rohdaten kontinuierlich angehängt. Das Dateisystem wächst dynamisch mit, es muss nichts vorallokiert werden. Die 24-Bit-Audiodaten werden in einem 3-Byte-Format abgelegt, was speicherplatzoptimal ist (keine 32-Bit-Verschwendung).

Fehlertoleranz: Verwerfen statt Abstürzen
Wenn der Schreib-Thread über längere Zeit nicht hinterherkommt (z.B. defekte SD-Karte), läuft der Ringpuffer voll. Anstatt den Echtzeit-Callback zu blockieren (was Dropouts verursachen würde), werden entweder die ältesten oder die neuesten Daten verworfen. Das ist ein bewusster Trade-off: Lieber eine Lücke in der Aufzeichnung als ein Absturz der gesamten Messung. Für ein "KANN"-Feature (optionale Aufzeichnung) ist dies akzeptabel – die primäre SPL-Berechnung läuft unbeeinträchtigt weiter.

Steuerung: Zustandsabhängig
Die Aufzeichnung ist ein optionaler Zustand innerhalb der Zustandsmaschine. Beim Übergang von "Messen" in "Aufzeichnen" wird ein Zeitstempel im Dateinamen hinterlegt. Beim Stoppen wird der WAV-Header finalisiert (die tatsächliche Dateigröße eingetragen) und die Datei geschlossen.

Das Ergebnis
Der Benutzer erhält eine zeitlich synchronisierte Audiodatei, die exakt der Messdauer entspricht (ohne Dropouts im Sinne von fehlenden Zeitabschnitten), während die SPL-Berechnung parallel in Echtzeit läuft. Die Dateigröße ist vorhersehbar: Bei 48 kHz, 24 Bit, mono sind das etwa 500 Megabyte pro Stunde Aufzeichnung.

REQ-FUNC-011

  tapy konnte nicht gefunden werden

Vielleicht "tappy" test anything library.

REQ-PERF-001

Technische Basis Pi Zero 2 W
CPU: Quad-Core ARM Cortex-A53 @ 1 GHz
RAM: 512 MB
I2S-Interface: Hardwareseitig bis 192 kHz möglich
Die Hardware kann 48 kHz/24-bit problemlos einlesen (nur 144 KB/s Datenrate). Das Problem ist die Echtzeit-Verarbeitung in Python.

Was funktioniert gut ✅
Reines Aufnehmen und Speichern:

Ohne komplexe Berechnungen läuft 48 kHz/24-bit stabil
sounddevice + numpy schafft das locker über Stunden
Einfache SPL-Berechnung (RMS):

Vektorisierte NumPy-Operationen auf Blöcken (1024-4096 Samples)
Block-basierte Filterung mit scipy.signal.lfilter
Was ist problematisch ⚠️
1. Python-Loops sind zu langsam Ein for-Loop über 4096 Samples in Python dauert Millisekunden – bei 48 kHz hat man nur 85 ms für den ganzen Block. Das klappt nicht für sampleweise A-Filter oder Oktavbandzerlegung.

2. Echtzeit-Prioritäten Linux (Raspberry Pi OS) ist kein Echtzeit-OS. Der Audio-Thread kann von anderen Prozessen (GUI, SSH, Systemdiensten) unterbrochen werden → Buffer-Underruns.

3. Terzband-Analyse (REQ-FUNC-005) Die Filterbank für Oktav-/Terzbänder in Python ist rechenintensiv. Bei 48 kHz mit 30 Terzbändern wird das knapp auf dem Pi Zero.

Konkrete Empfehlungen für erfolgreiche Umsetzung
Maßnahme	Warum
Blockgröße 2048-4096	~43-85 ms Pufferzeit, genug für NumPy-Operationen
NumPy/SciPy vektorisiert	Keine Python-Loops für Filter, RMS, Leq
Multiprocessing	Audio-Erfassung in separatem Prozess mit höherer Priorität (nice -n -10)
GUI entkoppelt	Streamlit/Web-GUI darf Audio-Thread nicht blockieren
Terzband optional prüfen	Ggf. reduzierte Bandanzahl oder Cython/Numba für Filter
Fazit
REQ-PERF-001 (48 kHz, keine Underruns) ist erreichbar, wenn:

Die Verarbeitung block-weise (nicht sample-weise) erfolgt
NumPy/SciPy für alle mathematischen Operationen genutzt wird
Der Audio-Prozess Priorität hat
Terzbandanalyse (REQ-FUNC-005) entweder optimiert oder mit reduzierter Abtastrate (z.B. 16 kHz nach Downsampling) berechnet wird
Risiko: REQ-FUNC-005 (Oktav-/Terzbandanalyse) + REQ-PERF-001 kombiniert ist auf dem Pi Zero 2 W die größte Herausforderung. Das sollte früh getestet werden.

ID: REQ-PERF-002

Was das bedeutet
Position	Wortbreite	Begründung
Extern (Mikrofon → Pi)	24 bit	INMP441 liefert 24-bit Auflösung in einem 32-bit I2S-Slot
Intern (Berechnung)	32 bit (float oder int)	Verhindert Überlauf bei Filterung und Pegelberechnung
Der INMP441 sendet 24 gültige Bits, aber der I2S-Bus transportiert 32 Bits pro Sample. Die unteren 8 Bits sind Nullen (Padding). Das ist das "24-bit in 32-bit Container" Format.

Umsetzung
Eingang (24-bit → 32-bit float): Das 24-bit Sample wird auf den Bereich [-1.0, 1.0] in 32-bit Float normiert:

Wert_float = Sample_int32 / 2^31
Beispiel:

Maximaler 24-bit Wert: 8.388.607 (0x7FFFFF) → ~0.9999999 in float32
Minimaler 24-bit Wert: -8.388.608 (0x800000) → ~-1.0 in float32
Verarbeitung (32-bit float):

Alle Filter (A-Bewertung, Zeitgewichtung) laufen in float32/float64
Zwischenergebnisse (Filterzustände, Energiesummen für Leq) werden in 32-bit gehalten
Kein Clipping bei dynamischen Berechnungen (float hat viel Headroom)
Ausgang (optional):

SPL-Werte in dB (float, 32-bit reicht)
JSON-Export (Textrepräsentation, keine Bittiefe relevant)
Warum diese Trennung?
24-bit extern: Das ist die maximale Auflösung des Mikrofons (INMP441). Mehr Bits wären sinnlos.
32-bit intern: Bei der Berechnung von Leq (Summe von Millionen von Samples) würde 24-bit integer überlaufen. Float32 verhindert das und bietet zusätzlich Dynamik für Zwischenergebnisse.
Verifikation/Akzeptanzkriterium
Test 1: 24-bit Eingabe sichtbar machen Ein Testsignal mit definierten 24-bit Werten (z.B. nur oberen 16 Bit gesetzt) wird eingespeist. Das System zeigt korrekte Pegel – wenn es nur 16-bit verarbeiten würde, wäre der Pegel falsch (6 dB zu niedrig).

Test 2: Kein Überlauf bei interner 32-bit Eine 5-Minuten-Messung mit Vollaussteuerung läuft ohne arithmetischen Überlauf (Leq-Berechnung summiert Millionen von Werten).

Test 3: Rauschabstand Das System unterscheidet zwischen -100 dB und -110 dB Signal (nur möglich mit echter 24-bit Auflösung, nicht mit 16-bit).

Zusammenfassung
Die Anforderung ist technisch trivial erfüllbar – das ist eher eine Dokumentation/Spezifikation des Datenflusses als eine komplexe Implementierungsaufgabe. Das I2S-Interface und sounddevice liefern nativ 32-bit Integer, man muss nur darauf achten, bei der Konvertierung zu float die volle 24-bit Dynamik zu erhalten (durch Division mit 2^31, nicht 2^23).

REQ-QUAL-001

Wir haben keine Pegelregelung, wenn wir keine Einbauen. Wir müssen prüfen, falls wir mit AI Coden, dass nicht zufällig eine Eingebaut wird. 


REQ-INTF-001

Die Anforderung ist Redundant. 

 REQ-NET-001

Wir haben uns für das Model Zero 2W entschieden. Damit ist die Anforderung bereits erfüllt. 

REQ-DATA-001

Ansatz 1: Separate Checksum-Datei (empfohlen)
Für jede exportierte Datei wird eine zusätzliche .sha256-Datei erstellt:

messung_20260511_143022.json
messung_20260511_143022.json.sha256
Vorteile:

Standardisiert (Linux-Tool sha256sum kann prüfen)
Dateiinhalt bleibt unverändert (keine circular dependency)
Einfach zu implementieren und zu verifizieren

REQ-DATA-002

Was die Anforderung bedeutet
Ziel: 12 Stunden Messdaten (SPL-Werte, keine Audioaufnahme) sollen auf dem Pi Zero gespeichert werden können, ohne dass der Speicher voll läuft. Optional soll die Speicherung komprimiert sein (Delta-Encoding).

Realitätscheck:

Pi Zero hat typischerweise eine 16-32 GB SD-Karte
500 MB freier Speicher sind also kein Problem (das ist nur ~1-3% der Karte)
Datenvolumen-Berechnung
Szenario: 12h Messung, 10 Hz Update-Rate (alle 100ms ein Wert)

Parameter	Wert
Dauer	12 h = 43.200 s
Datenpunkte	432.000 (bei 10 Hz)
Daten pro Punkt (JSON)	~80-100 Bytes
Rohdaten	~35-40 MB
Mit Kompression	~10-15 MB
Fazit: Selbst ohne Delta-Encoding passen 12h problemlos in 500 MB.

REQ-PWR-001

Das ist kein Problem. Kabel rein, fertig. Vielleicht kann man eine kleine Powerback vorschalten. 

REQ-MECH-001

Das Gehäuse kann ich (Lars) einfach designen und ausdrucke.

REQ-UI-001

Wir haben kein Display am Gerät und bekommen auch keins. Das ist einfach zu erfüllen. 

REQ-MIC-001

Das Mikrophon hat keine Richtcharakteristik.

REQ-STD-001

REQ-FUNC-003 sagt was gebaut wird. REQ-STD-001 fordert den Nachweis, dass es richtig gebaut wurde.


REC-LIC-001

Das Projekt wird unter MIT-Lizenz veröffentlicht, da dies als universitäres Lehrprojekt maximalen Wert durch Verbreitung und Nachnutzung schafft. Die MIT-Lizenz ermöglicht Studierenden, Industriepartnern und anderen Hochschulen eine uneingeschränkte Nutzung – kommerziell wie akademisch – ohne rechtliche Hürden. Als Lehrprojekt steht der Wissenstransfer und nicht der kommerzielle Schutz im Vordergrund. Die Lizenz garantiert Attribution für die TU Berlin als Urheber, erfordert aber keine Offenlegung abgeleiteter Werke. Dies fördert Innovation und maximale Adaption des entwickelten SPL-Meter-Systems.

REQ-VAL-001

Wir geben das Gerät an Jakob und lassen die ergebnisse Validieren.


 REQ-ARCH-001

Empfohlen: transitions Library

Industriestandard für Python-Zustandsmaschinen
UML-konforme Zustandsdiagramme automatisch generierbar
Unterstützt Hierarchische Zustände (für Substates)
Thread-safe Optionen verfügbar
Dokumentation: https://github.com/pytransitions/transitions
Alternative: python-statemachine

Leichtgewichtiger, moderner API-Design
Gut für einfache Zustandsmaschinen ohne komplexe Hierarchien
Minimal-Lösung: Python enum + Callbacks

Keine externe Abhängigkeit
Manuelle Übergangslogik
Gut wenn das Projekt sehr schlank bleiben soll
Visualisierung / Dokumentation
Empfohlen: PlantUML (wird bereits im Projekt verwendet)

Ihr habt schon spl_meter_statechart.puml im W3-Ordner
Textbasiert → versionierbar im Git
Gradle/CI-Integration möglich
Rendert zu PNG/SVG für Dokumentation

Empfehlung für euer Projekt:

PlantUML für Dokumentation/Pflichtenheft (bereits vorhanden)
transitions Library für Implementation (mächtig, gut dokumentiert)
pytest für Test der Zustandslogik

REQ-DIAG-001

Konzept: Zustandsabhängige Selbsttests
Jeder Substate (feiner Zustand innerhalb der Hauptzustände) hat eine zugeordnete Diagnosefunktion. Diese prüft die spezifischen Ressourcen, die in diesem Zustand benötigt werden.

Beispiel-Zuordnung
Substate	Diagnosefunktion	Was wird geprüft?
Boot → Hardware-Init	test_hardware()	I2S-Interface erreichbar? Kernel-Module geladen?
Idle → Bereit	test_standby()	Mikrofon liefert Daten (kurzer Test-Read)? SD-Karte beschreibbar?
Messen → Läuft	test_audio_stream()	Keine Underruns seit Start? Pegel im erwarteten Bereich (nicht dauerhaft 0)?
Aufzeichnen → Schreibt	test_storage()	Schreibgeschwindigkeit ausreichend? Datei wächst?
Kalibrieren → Referenz	test_calibration_signal()	1kHz-Kalibrator-Signal erkannt? Amplitude im Fenster?


1. Test-Pyramide (Verhältnis 70/20/10)
Unit-Tests (70%) – Reine Logik ohne Hardware

Mathematische Funktionen: SPL-Berechnung, Zeitgewichtungen, Leq-Integration
Filterkoeffizienten: A-Bewertung Frequenzgang prüfen
Zustandsmaschine: Übergänge mit Mock-Objekten testen
Integrationstests (20%) – Module zusammen

Audio-Pipeline: Von Rohdaten → gefiltert → SPL-Wert
JSON-Export: Datenstruktur + Checksumme validieren
API: HTTP-Endpunkte mit Test-Client
System/Hardware-Tests (10%) – Am realen Pi

5-Minuten-Stabilitätstest (REQ-FUNC-001)
Vergleich NTi XL2 (REQ-VAL-001)
Buffer-Underrun-Test unter Last (SSH + GUI gleichzeitig)
2. Test-Driven für kritische Funktionen
Welche zuerst? (Priorisierung nach Risiko)

SPL-Berechnung (REQ-FUNC-002) – Fundament, mathematisch prüfbar
Test: Sinus mit bekannter Amplitude → erwarteter dB-Wert
Test: Null-Signal → -inf dB (kein Crash)
Zeitgewichtungen (REQ-FUNC-003) – IEC-Konformität kritisch
Test: Sprungantwort → 63% nach τ (Toleranz ±1%)
Test: Leq über 60s mit konstantem Signal → identisch zum Eingang
Frequenzbewertung (REQ-FUNC-004) – Filter korrekt?
Test: 1kHz Sinus → 0 dB Dämpfung (A-Bewertung)
Test: 100Hz Sinus → ~-19 dB Dämpfung

 REQ-ERR-001

Fehlerklassen und Reaktionen
Fehler	Erkennung	Degradation	Recovery
Mikrofon getrennt	sounddevice PortAudioError bei Stream-Start	Zustand → Fehler, GUI zeigt "Hardwarefehler"	Benutzer muss Kabel prüfen, dann "Retry" oder Neustart
Übersteuerung (Clipping)	Sample-Wert ≥ 0 dBFS (max int32)	Log-Warnung, gelber Indikator in GUI, Messung läuft weiter	Automatisch wenn Pegel sinkt
Buffer-Underrun	status.input_underflow im Callback	Zähler erhöhen, nach 3x → Zustand "Fehler"	Neustart der Messung (Zustand → Idle → Messen)
SD-Karte voll	OSError: No space left on device	Messung pausieren, Warnung ausgeben	Benutzer löscht Daten, dann Fortsetzung




ID: REQ-PROC-001

Abriss REQ-PROC-001 – Zweiwöchige Iterationen
Ziel: Alle 2 Wochen ein funktionsfähiges Inkrement + Review-Meeting. Jeder Review umfasst 10 Minuten Präsentation (1 Person), 7–8 Folien, Stand der Anforderungen, Probleme, nächstes Vorgehen.
Termine aus W5/termine_und_daten.md
1. Zwischenbericht/Präsentation: 18.05. (Inhalte: 10 min, 7–8 Slides; Einführung, bisheriges Vorgehen, Anforderungsstand, Probleme, weiteres Vorgehen).
Semesterende: 30.06.2026.
Abgabe (Moodle-Aufgabe): Fällig 04.07.2026, 00:01.
Fortschrittsberichte: 10 min, 1× pro Person, alle 2–3 Wochen.
Iterationsplan (Vorschlag)
Sprint A (Kurz, Vorab) – bis 18.05.
Fokus: Anforderungen konsolidieren (Anforderungen.md), Zustandsmaschine (zustandsmaschine.md), Architektur-Entwurf.
Review: 18.05. Präsentation (Zwischenbericht #1).
Inkrement: Doku + lauffähiges Repo-Skelett (Build, Lint, Minimal-Start der GUI).
Sprint 1 – 18.05. bis 01.06. (Review 01.06.)
Inhalte: Digitaler Audioeingang (USB/I2S) lauffähig; Echtzeit-Pipeline-Skelett; Web-GUI Grundfunktionen; JSON-Grundschema.
Inkrement: v0.1 Tag/Release; Demo: Live-Pegel (roh), Start/Stop.
Sprint 2 – 01.06. bis 15.06. (Review 15.06.)
Inhalte: Zeitbewertungen F/S + Leq; Frequenzbewertungen A/Z; erste Stabilitätstests (48 kHz/24 bit).
Inkrement: v0.2 Tag/Release; Demo: korrekte Kennwerte, flüssige GUI.
Sprint 3 – 15.06. bis 29.06. (Review 29.06.)
Inhalte: Oktav-/Terzbänder; Kalibrierung 94 dB @ 1 kHz; Checksummen im Export; Basis-CLI/SSH; Fehlerhandling (Device lost/Underruns).
Inkrement: v0.3 Tag/Release; Demo: validierte Bandpegel, Kalibrierschritt, Export mit Prüfsumme.
Sprint 4 (Abschluss) – 29.06. bis 04.07.
Inhalte: Vergleich gegen Referenz (NTi XL2) mit Toleranzen; Doku-Finalisierung; Packaging/Installer/Startskripte.
Inkrement: v1.0 Tag/Release; Abgabe bis 04.07., 00:01.
Review-Organisation
Dauer/Format: 10 min Vortrag + kurze Diskussion, 7–8 Folien.
Rollen: Jedes Review präsentiert eine andere Person (4 Personen → 4 Reviews: 18.05., 01.06., 15.06., 29.06.).
Inhalte pro Review:
Stand der Anforderungen (Delta/Traceability).
Funktionsdemo des Inkrements (Video/GIF/Live).
Mess-/Testnachweise (kurz: Screens/Plots).
Risiken/Nächste Schritte + klarer Sprint-Backlog.